# 08주 · 개수로 기하·유량 측정

수리학 CE272 · 주차별 공통 실습

핵심: 원리·식의 의미·손계산·조건 변경·결과 해석. 입력 단위와 지배 방정식·경계조건부터 확인

학습 주제: 개수로 흐름의 분류 / 단면·유속 분포·유량 측정

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/week08/CE272_W08.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE272-hydraulics/week08) · [계산 도구·가정](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/METHODS.md)

[공통 학습 계획](https://github.com/steelpl/lecture/blob/main/CE272-hydraulics/COURSE_PLAN.md) · [학기 운영안내](https://github.com/steelpl/lecture/blob/main/CURRENT_OFFERING.md)

실행: 처음에는 위에서 아래로 셀 실행. 공통 준비 셀 실행 후 각 S절 개별 실행 가능

진행: 예측 → 손계산·식 설정 → 실행 → 조건 하나 변경 → 원리·가정·결과 설명

Excel 중점: 단면 기하·유량 측정

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import importlib.util
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.integrate import trapezoid

# 로컬 공통 자료에서는 옆 과목 파일 사용. Colab에서는 같은 공개 모듈 불러오기
module_path = next((p for p in [Path.cwd()/"hydraulics.py", Path.cwd().parent/"hydraulics.py"] if p.exists()), None)
if module_path is None:
    module_path = Path(tempfile.gettempdir())/"lecture_ce272_hydraulics.py"
    urlretrieve("https://raw.githubusercontent.com/steelpl/lecture/main/CE272-hydraulics/hydraulics.py",module_path)
spec = importlib.util.spec_from_file_location("lecture_ce272_hydraulics",module_path)
hyd = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hyd)
print("수리학 공통 계산 도구 준비 완료. SI 단위·Darcy 계수 사용")

## S01. 사다리꼴 단면과 Froude 수

**슬라이드 연결**: CE272_W08a_Open_Channel_Flow_1.pptx p11, 12, 13 / CE272_W08b_Open_Channel_Flow_2.pptx p9, 10, 11, 12

**자료 구분**: 개념 적용·추가 조건

**먼저 예측**: Froude 수의 수심에 Rh를 대신 넣어도 되는가?

**문제**: 바닥 폭 2 m, 측면경사 z=2, 수심 1 m인 사다리꼴 단면의 A, P, Rh, 수면 폭 T와 동수수심 A/T를 구하시오. Q=4 m³/s에서 흐름을 분류하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
y,b,z,Q = 1.,2.,2.,4.
s=hyd.section(y,b,z)
pd.Series({**s,'hydraulic_depth_A_T':s['A']/s['T'],
           'V_m_s':Q/s['A'],'Froude':hyd.froude(Q,y,b,z)})

**검토 문제**: Manning 식과 Froude 수에서 서로 다른 기하량을 사용하는 이유를 설명하시오. 등류가 단면 전체의 동일한 유속을 뜻하는지도 검토하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## S02. 2점 유속과 단면 유량

**슬라이드 연결**: CE272_W08b_Open_Channel_Flow_2.pptx p14, 15, 16

**자료 구분**: 원본 측정법·추가 관측 자료

**먼저 예측**: 단면 전체의 평균유속 하나와 각 측선의 평균유속은 같은 자료인가?

**문제**: 아래 추가 관측 자료에서 0.2·0.8 수심 유속의 평균으로 측선 평균유속을 구하시오. 평균단면법과 중앙단면법의 유량을 계산하고 두 결과가 다른 이유를 설명하시오.

예상값·단위·식·가정을 먼저 기록한 뒤 아래 계산과 비교

In [ ]:
x=np.array([0.,2.,4.,6.,8.])
depth=np.array([0.,1.2,2.,1.5,0.])
v02=np.array([0.,.6,.9,.7,0.]);v08=np.array([0.,.4,.6,.5,0.])
vbar=(v02+v08)/2
Q_mean=np.sum(np.diff(x)*(depth[:-1]+depth[1:])/2*(vbar[:-1]+vbar[1:])/2)
Q_mid=np.sum(depth[1:-1]*vbar[1:-1]*(x[2:]-x[:-2])/2)
display(pd.DataFrame({'x_m':x,'depth_m':depth,'v_02':v02,'v_08':v08,'vbar':vbar}))
print('Mean-section Q:',Q_mean,'; mid-section Q:',Q_mid,'m³/s')

**검토 문제**: 측선 간격을 좁히면 두 추정값의 차이가 항상 같은 방향으로 줄어드는지 검토하시오. 관측 오차와 구간 근사의 오차를 구분하시오.

검산: 단위·보존식 잔차·독립 계산 확인. 모형·반올림·이산화 오차 구분

## 학습 기록

동수수심·동수반경을 구분하고 관측 방식이 유량 추정에 미치는 영향을 설명하시오.

선택 심화: 개념 학습 후 AI의 설명·계산 검토. 단위·가정·보존식으로 확인한 근거와 수정한 부분 기록

In [ ]:
prediction = None
hand_calculation = None
changed_condition = None
principle_and_limits = None
records = {'예측':prediction,'손계산':hand_calculation,'조건 변경':changed_condition,'원리·한계':principle_and_limits}
missing = [k for k,v in records.items() if v is None or str(v).strip() == '']
print('작성 필요:',missing) if missing else print('기록 항목 작성 완료. 계산·논리는 별도 검토 필요')